### data test

In [ ]:
import os
import random
import matplotlib.pyplot as plt
from PIL import Image

TRAIN_DIR = r"D:\sup_ai\parasite-cv\cnn_dataset_v3\train"
classes = [str(i) for i in range(11)]

plt.figure(figsize=(18, 6))
plt.suptitle("Random Sample from Each Class", fontsize=16, fontweight='bold', y=1.05)

for i, cls in enumerate(classes):
    class_path = os.path.join(TRAIN_DIR, cls)
    
    if os.path.exists(class_path):
        images = os.listdir(class_path)
        if images:
            rand_img = random.choice(images)
            img_path = os.path.join(class_path, rand_img)
            img = Image.open(img_path)
            
            plt.subplot(2, 6, i + 1)
            plt.imshow(img)
            plt.title(f"Class {cls}\n({img.size[0]}x{img.size[1]})", fontsize=10)
            plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

TRAIN_DIR = r"D:\sup_ai\parasite-cv\cnn_dataset_v3\train"
VAL_DIR = r"D:\sup_ai\parasite-cv\cnn_dataset_v3\val" 

data_counts = []

for cls in range(11):
    train_count = 0
    val_count = 0
    
    train_path = os.path.join(TRAIN_DIR, str(cls))
    if os.path.exists(train_path):
        train_count = len(os.listdir(train_path))
        
    val_path = os.path.join(VAL_DIR, str(cls))
    if os.path.exists(val_path):
        val_count = len(os.listdir(val_path))
    else:
        val_count = int(train_count * 0.2)
        train_count = train_count - val_count
        
    data_counts.append({"Class": f"Class {cls:02d}", "Split": "Train", "Count": train_count})
    data_counts.append({"Class": f"Class {cls:02d}", "Split": "Validation", "Count": val_count})

df_counts = pd.DataFrame(data_counts)

plt.figure(figsize=(15, 6))
sns.barplot(data=df_counts, x="Class", y="Count", hue="Split", palette=["#2ecc71", "#e74c3c"])
plt.title("Data Imbalance Check: Train vs Validation Distribution", fontsize=16, fontweight='bold')
plt.ylabel("Number of Images", fontsize=12)
plt.xlabel("Parasite Class", fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title="Dataset Split")
plt.show()
plt.figure(figsize=(12, 4))
pivot_df = df_counts.pivot(index="Split", columns="Class", values="Count")

sns.heatmap(pivot_df, annot=True, fmt="d", cmap="Blues", linewidths=1, linecolor='white')
plt.title("2D Matrix: Image Counts per Class (Train vs Val)", fontsize=16, fontweight='bold', pad=15)
plt.yticks(rotation=0)
plt.show()

### yolo test [ prepare ]

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

yolo_model_path = r"D:\sup_ai\parasite-cv\runs\detect\yolo_localizer\weights\best.pt"
test_dir = Path(r"D:\sup_ai\parasite-cv\test_set\my_test")
model = YOLO(yolo_model_path)

valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}

image_files = [f for f in test_dir.iterdir() if f.is_file() and f.suffix.lower() in valid_extensions]

if not image_files:
    print(f"No image files found in {test_dir}")
else:
    print(f"Image: {len(image_files)} In processing\n")
    print("-" * 50)

for img_path in image_files:
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        continue
    
    results = model(img_bgr, verbose=False)
    boxes = results[0].boxes.xyxy.cpu().numpy()
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    if len(boxes) == 0:
        print(f"Image: {img_path.name} | No parasites/cells found in this image")
        continue
        
    print(f"Image: {img_path.name} | YOLO Found {len(boxes)} bounding box(es)")
    
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box)
        
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(img_rgb.shape[1], x2), min(img_rgb.shape[0], y2)
        
        if x2 <= x1 or y2 <= y1:
            continue
            
        img_with_box = img_rgb.copy()
        cv2.rectangle(img_with_box, (x1, y1), (x2, y2), (255, 0, 0), 3)
        
        cropped_img = img_rgb[y1:y2, x1:x2]
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        
        axes[0].imshow(img_with_box)
        axes[0].set_title(f"Original + Bounding Box (Detection {i+1})")
        axes[0].axis('off')
        
        axes[1].imshow(cropped_img)
        axes[1].set_title(f"Cropped Result")
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.show()

### cnn-automl

In [ ]:
import os
import cv2
import shutil
import albumentations as A
from glob import glob
from tqdm import tqdm

In [ ]:
INPUT_ROOT = r"D:\sup_ai\parasite-cv\Chula-ParasiteEgg-11\Chula-ParasiteEgg-11\data"
OUTPUT_ROOT = r"D:\sup_ai\parasite-cv\Chula-ParasiteEgg-11_Augmented\data"

In [ ]:
transform = A.Compose([
    A.CLAHE(clip_limit=8.0, tile_grid_size=(8, 8), p=0.5),
    A.Sharpen(alpha=(0.2, 0.5), lightness=(0.5, 1.0), p=0.4),
    A.GaussNoise(var_limit=(10.0, 50.0), mean=0, p=0.4),
    A.PixelDropout(dropout_prob=0.02, p=0.3),
    A.RandomFog(fog_coef_lower=0.03, fog_coef_upper=0.1, alpha_coef=0.4, p=0.3),
    A.Blur(blur_limit=(5, 7), p=0.3)
])

In [ ]:
def process_full_dataset(input_root, output_root, augment_multiplier=2):
    # รองรับนามสกุลภาพทั่วไป
    valid_extensions = ('.png', '.jpg', '.jpeg', '.JPG', '.PNG')
    
    print(f"กำลังสแกน Dataset จาก: {input_root}")
    
    for root, dirs, files in os.walk(input_root):
        # คัดเฉพาะไฟล์รูปภาพ
        image_files = [f for f in files if f.endswith(valid_extensions)]
        
        if not image_files:
            continue # ถ้าโฟลเดอร์นี้ไม่มีรูป ให้ข้ามไป
            
        # สร้าง Path ปลายทางให้โครงสร้างเหมือนต้นฉบับเป๊ะๆ
        rel_path = os.path.relpath(root, input_root)
        target_dir = os.path.join(output_root, rel_path)
        os.makedirs(target_dir, exist_ok=True)
        
        # เช็คว่าโฟลเดอร์นี้เป็น Train หรือไม่ (เช็คจากชื่อ Path)
        # ถ้าใช่ = ทำ Augmentation / ถ้าไม่ใช่ (Val, Test) = ก๊อปปี้ไปดื้อๆ
        is_train_folder = 'train' in root.lower()
        
        mode_text = "[Train] ทำ Augment" if is_train_folder else "[Val/Test] ก๊อปปี้ปกติ"
        print(f"\n{mode_text} -> {rel_path} ({len(image_files)} รูป)")
        
        for filename in tqdm(image_files, leave=False):
            img_path = os.path.join(root, filename)
            target_original_path = os.path.join(target_dir, filename)
            
            # 1. ก๊อปปี้ภาพต้นฉบับไปไว้ที่โฟลเดอร์ใหม่ (ต้องมีเสมอ)
            shutil.copy2(img_path, target_original_path)
            
            # 2. ถ้าเป็น Train folder ให้ปั๊มภาพจำลองเพิ่ม
            if is_train_folder:
                image = cv2.imread(img_path)
                if image is None: continue
                
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                name, ext = os.path.splitext(filename)
                
                for i in range(augment_multiplier):
                    augmented = transform(image=image)
                    aug_image = augmented['image']
                    
                    aug_filename = f"{name}_aug_{i}{ext}"
                    aug_target_path = os.path.join(target_dir, aug_filename)
                    cv2.imwrite(aug_target_path, cv2.cvtColor(aug_image, cv2.COLOR_RGB2BGR))

    print("\nประมวลผล Dataset เสร็จสมบูรณ์!")
    print(f"ข้อมูลใหม่ทั้งหมดถูกเซฟไว้ที่: {output_root}")

# เริ่มทำงาน! ปั๊มรูปเพิ่ม 2 เท่า (รวมของเดิมจะเป็น 3 เท่า)
if __name__ == "__main__":
    process_full_dataset(INPUT_ROOT, OUTPUT_ROOT, augment_multiplier=2)

### json to yolo txt

In [ ]:
import json
import os

# --- ตั้งค่า Path ---
json_path = r"D:\sup_ai\parasite-cv\Chula-ParasiteEgg-11\Chula-ParasiteEgg-11\labels.json"
output_data_root = r"D:\sup_ai\parasite-cv\Chula-ParasiteEgg-11_Augmented\data"

def convert_to_yolo():
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 1. สร้าง Dictionary เก็บข้อมูลรูปภาพ และตัวช่วยค้นหา ID จากชื่อไฟล์
    images_by_id = {img['id']: img for img in data['images']}
    filename_to_id = {os.path.basename(img['file_name']): img['id'] for img in data['images']}

    # 2. สร้าง Dictionary เก็บ Bbox ที่แปลงเป็น YOLO Format แล้ว
    annotations = {}
    for ann in data['annotations']:
        img_id = ann['image_id']
        
        # ป้องกันกรณีหา ID ไม่เจอ
        if img_id not in images_by_id:
            continue
            
        img_w = images_by_id[img_id]['width']
        img_h = images_by_id[img_id]['height']
        
        x_min, y_min, w, h = ann['bbox']
        x_center = (x_min + w / 2) / img_w
        y_center = (y_min + h / 2) / img_h
        nw = w / img_w
        nh = h / img_h
        
        if img_id not in annotations: 
            annotations[img_id] = []
        annotations[img_id].append(f"{ann['category_id']} {x_center:.6f} {y_center:.6f} {nw:.6f} {nh:.6f}")

    print("กำลังสร้างไฟล์ Label (.txt) ให้ตรงกับรูปภาพที่ Augment มา...")
    
    # 3. ไล่หาไฟล์ภาพแล้วจับคู่สร้าง .txt ไว้ข้างๆ กัน
    count = 0
    for root, dirs, files in os.walk(output_data_root):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                
                # คืนร่างชื่อไฟล์ต้นฉบับ (แก้บั๊กนามสกุลไฟล์หาย)
                if '_aug_' in file:
                    name_part = file.split('_aug_')[0]
                    ext_part = os.path.splitext(file)[1]
                    original_filename = name_part + ext_part
                else:
                    original_filename = file
                
                # หา image_id จากชื่อไฟล์
                img_id = filename_to_id.get(original_filename)
                
                # ถ้าเจอข้อมูล Bbox ก็สร้างไฟล์ .txt ได้เลย
                if img_id and img_id in annotations:
                    txt_path = os.path.join(root, os.path.splitext(file)[0] + ".txt")
                    with open(txt_path, 'w') as f:
                        f.write("\n".join(annotations[img_id]))
                    count += 1
    
    print(f"✅ สร้างไฟล์ Label เสร็จแล้วทั้งหมด {count} ไฟล์!")

if __name__ == "__main__":
    convert_to_yolo()

### yolov8

In [ ]:
import os
import shutil
import random
from tqdm import tqdm  # เพิ่มไลบรารีสำหรับทำหลอดโหลด

SOURCE_DIR = r"D:\sup_ai\parasite-cv\Chula-ParasiteEgg-11_Augmented\data"
OUTPUT_DIR = r"D:\sup_ai\parasite-cv\Chula-ParasiteEgg-11_YOLO_Ready"

def split_and_prepare_yolo_dataset():
    print("กำลังเตรียมโครงสร้างโฟลเดอร์ Train/Val ให้ YOLOv8...\n")
    
    # 1. สร้างโฟลเดอร์
    for split in ['train', 'val']:
        os.makedirs(os.path.join(OUTPUT_DIR, split, 'images'), exist_ok=True)
        os.makedirs(os.path.join(OUTPUT_DIR, split, 'labels'), exist_ok=True)
        
    # 2. กวาดรายชื่อรูปภาพทั้งหมด
    all_images = [f for f in os.listdir(SOURCE_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.seed(42)
    random.shuffle(all_images)
    
    # 3. แบ่ง Train 80% / Val 20%
    split_index = int(len(all_images) * 0.8)
    train_images = all_images[:split_index]
    val_images = all_images[split_index:]
    
    # ฟังก์ชันก๊อปปี้ไฟล์พร้อมแสดง Progress Bar
    def copy_data(image_list, split_name):
        count = 0
        # ใช้ tqdm ครอบ image_list เพื่อสร้างหลอดโหลด
        for img_name in tqdm(image_list, desc=f"ก๊อปปี้ไฟล์เข้า {split_name.upper()}", unit="ไฟล์"):
            src_img = os.path.join(SOURCE_DIR, img_name)
            txt_name = os.path.splitext(img_name)[0] + '.txt'
            src_txt = os.path.join(SOURCE_DIR, txt_name)
            
            dst_img = os.path.join(OUTPUT_DIR, split_name, 'images', img_name)
            dst_txt = os.path.join(OUTPUT_DIR, split_name, 'labels', txt_name)
            
            if os.path.exists(src_txt):
                shutil.copy2(src_img, dst_img)
                shutil.copy2(src_txt, dst_txt)
                count += 1
        return count

    print(f"พบรูปภาพทั้งหมด {len(all_images)} รูป (พร้อมไฟล์ Label)")
    train_count = copy_data(train_images, 'train')
    val_count = copy_data(val_images, 'val')
    
    print(f"\nสร้าง Dataset เสร็จสมบูรณ์ที่: {OUTPUT_DIR}")
    print(f"   -> Train: {train_count} รูป")
    print(f"   -> Val: {val_count} รูป")

if __name__ == "__main__":
    split_and_prepare_yolo_dataset()

In [ ]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

print("🧹 ล้างแรมการ์ดจอเรียบร้อยแล้ว!")

In [ ]:
from ultralytics import YOLO

def train_yolov8_paper_specs():
    # โหลดโมเดล Object Detection 
    model = YOLO('yolov8m.pt') 

    print("เริ่มต้นกระบวนการ Training Object Detection...")
    print("(YOLOv8 จะแสดง Progress Bar โดยอัตโนมัติเมื่อเริ่มรัน Epoch ที่ 1)")

    results = model.train(
        data=r'D:\sup_ai\parasite-cv\data.yaml', 
        epochs=200,
        patience=50,
        
        # --- 🔴 จุดที่แก้เพื่อกันเครื่องค้าง ---
        batch=8,      # ลดจาก 16 เหลือ 4 (ถ้ายังค้างให้ลดเหลือ 2)
        workers=0,    # จำกัดการใช้ CPU (ถ้ายังค้างให้ลดเหลือ 0 หรือ 1)
        cache=False,  # ห้ามโหลดรูปเก็บไว้ใน RAM รวดเดียว
        # ------------------------------------
        
        imgsz=640,
        optimizer='SGD',
        lr0=0.01,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        cos_lr=True,
        warmup_epochs=3.0,
        warmup_momentum=0.8,
        
        # ปิด Augment ตามเดิม
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, degrees=0.0, translate=0.0,
        scale=0.0, shear=0.0, perspective=0.0, flipud=0.0, fliplr=0.5,
        mosaic=0.0, mixup=0.0,
        
        device=0
    )
    
if __name__ == "__main__":
    train_yolov8_paper_specs()

### dinov3

In [ ]:
import os
import cv2
from ultralytics import YOLO
from tqdm import tqdm

def prepare_dino_dataset(model_path, images_dir, labels_dir, output_dir):
    model = YOLO(model_path)
    
    # สร้างโฟลเดอร์สำหรับแต่ละคลาส (0-10)
    for i in range(11):
        os.makedirs(os.path.join(output_dir, str(i)), exist_ok=True)

    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
    
    print("✂️ กำลังเริ่มตัดภาพ (Cropping) เพื่อเตรียมเทรน DINO...")
    for img_name in tqdm(image_files):
        img_path = os.path.join(images_dir, img_name)
        img = cv2.imread(img_path)
        if img is None: continue
        
        # ใช้ YOLO ทำนายเพื่อหาตำแหน่งกล่อง (หรือจะใช้ Ground Truth จาก Label เลยก็ได้)
        results = model.predict(img, conf=0.25, verbose=False)
        
        for i, box in enumerate(results[0].boxes):
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            cls = int(box.cls[0].item())
            
            # ตัดรูปเฉพาะส่วนที่เป็นไข่
            crop = img[y1:y2, x1:x2]
            
            if crop.size > 0:
                # เซฟรูปลงในโฟลเดอร์ของคลาสนั้นๆ
                save_path = os.path.join(output_dir, str(cls), f"{img_name.split('.')[0]}_box{i}.jpg")
                cv2.imwrite(save_path, crop)

if __name__ == "__main__":
    # 🔴 ปรับ Path ของไฟล์ในเครื่องคุณ
    MY_MODEL = r"D:\sup_ai\parasite-cv\runs\detect\train-11\weights\best.pt"
    TRAIN_IMAGES = r"D:\sup_ai\parasite-cv\Chula-ParasiteEgg-11\Chula-ParasiteEgg-11\data"
    # โฟลเดอร์ที่จะเก็บรูปสำหรับไปเทรน DINO
    DINO_DATA_DIR = r"D:\sup_ai\parasite-cv\dino_training_data"
    
    prepare_dino_dataset(MY_MODEL, TRAIN_IMAGES, None, DINO_DATA_DIR)

In [ ]:
!pip install torchmetrics

In [ ]:
!pip install termcolor

In [ ]:
import torch

# สั่งให้ PyTorch ลิสต์รายชื่อโมเดลทั้งหมดใน DINOv3 ออกมา
available_models = torch.hub.list('facebookresearch/dinov3')
print("รายชื่อโมเดลที่โหลดได้:", available_models)

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm
import timm

# ==========================================
# 1. สถาปัตยกรรมโมเดล DINOv2 (ผ่าน timm)
# ==========================================
class DINOClassifier(nn.Module):
    def __init__(self, num_classes=11, freeze_backbone=True):
        super().__init__()
        print(f"📥 กำลังโหลด DINOv2 (ViT-Small) จาก timm...")
        
        # ใช้โมเดล vit_small_patch14_dinov2 ซึ่งเป็นตัวที่แรงและเบา
        self.backbone = timm.create_model('vit_small_patch14_dinov2.lvd142m', pretrained=True)
        
        # แช่แข็ง Backbone เพื่อเทรนเฉพาะส่วนหัว (Fine-tuning)
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
        
        # ดึงจำนวน Feature จากตัวโมเดลต้นทาง (ปกติคือ 384 สำหรับ vit_small)
        embed_dim = self.backbone.num_features
        
        # สร้างส่วน Classifier ใหม่ให้ตรงกับจำนวนคลาสของคุณ
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        # timm backbone จะคืนค่า global pool features มาให้เลย
        features = self.backbone(x)
        return self.classifier(features)

# ==========================================
# 2. ฟังก์ชันช่วยเรียงลำดับ Class ให้ถูกต้องตามตัวเลข (0, 1, 2... 10)
# ==========================================
def get_numeric_class_mapping(directory):
    # ดึงชื่อโฟลเดอร์ทั้งหมดและกรองเฉพาะที่เป็นตัวเลข
    folders = [f for f in os.listdir(directory) if os.path.isdir(os.path.join(directory, f))]
    # เรียงลำดับตามค่าตัวเลข (เพื่อให้ 10 อยู่หลัง 9)
    sorted_folders = sorted(folders, key=lambda x: int(x) if x.isdigit() else x)
    # สร้าง dictionary สำหรับจับคู่: {'0': 0, '1': 1, ..., '10': 10}
    class_to_idx = {folder: i for i, folder in enumerate(sorted_folders)}
    return class_to_idx

# ==========================================
# 3. ฟังก์ชันการฝึกสอน (Training Loop)
# ==========================================
def train_model(model, train_loader, epochs=10, lr=1e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    # เทรนเฉพาะน้ำหนักในส่วน classifier เท่านั้น
    optimizer = optim.AdamW(model.classifier.parameters(), lr=lr, weight_decay=1e-2)
    scaler = GradScaler()

    print(f"🚀 เริ่มต้นการเทรนบนอุปกรณ์: {device}")
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            
            with autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            pbar.set_postfix({
                'Loss': f"{running_loss/len(train_loader):.4f}",
                'Acc': f"{100.*correct/total:.2f}%"
            })
            
    torch.save(model.state_dict(), "dino_v2_timm_best.pth")
    print(f"\n✅ บันทึกโมเดลเรียบร้อย: 'dino_v2_timm_best.pth'")
    return model

# ==========================================
# 4. ส่วนการทำงานหลัก (Main)
# ==========================================
if __name__ == "__main__":
    DATA_DIR = r"D:\sup_ai\parasite-cv\dino_training_data"
    BATCH_SIZE = 16 # RTX 4060 รับไหวสบายๆ สำหรับรูปเดียว
    EPOCHS = 15

    # 1. การเตรียมรูปภาพ (Data Augmentation)
    data_transforms = transforms.Compose([
        transforms.Resize((518, 518)), # <--- เปลี่ยนเป็น 518 ตรงนี้
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    # 2. แก้ปัญหาการเรียงลำดับ Index
    custom_class_to_idx = get_numeric_class_mapping(DATA_DIR)
    
    # สร้าง Dataset โดยบังคับใช้ mapping ที่เราสร้างเอง
    train_dataset = datasets.ImageFolder(root=DATA_DIR, transform=data_transforms)
    train_dataset.class_to_idx = custom_class_to_idx
    # แก้ไข samples ให้ตรงกับ mapping ใหม่ (สำคัญมาก)
    train_dataset.samples = [(path, custom_class_to_idx[os.path.basename(os.path.dirname(path))]) 
                             for path, _ in train_dataset.samples]

    print(f"📊 ตรวจสอบการจับคู่คลาส:")
    for name, idx in train_dataset.class_to_idx.items():
        print(f"   โฟลเดอร์ '{name}' -> Label Index: {idx}")

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

    # 3. โหลดโมเดลและเริ่มเทรน
    num_classes = len(train_dataset.classes)
    model = DINOClassifier(num_classes=num_classes, freeze_backbone=True)
    
    train_model(model, train_loader, epochs=EPOCHS)

    # 4. บันทึกชื่อคลาสไว้ใช้ตอนทายผล (Inference)
    with open("class_names.txt", "w") as f:
        # เรียงตาม index 0, 1, 2...
        names_in_order = sorted(custom_class_to_idx.items(), key=lambda x: x[1])
        for name, _ in names_in_order:
            f.write(f"{name}\n")
    print("📝 บันทึกรายชื่อคลาสลงใน 'class_names.txt' แล้ว")

### post-process

In [ ]:
import os
import cv2
import pandas as pd # เพิ่ม pandas สำหรับอ่าน csv
from ultralytics import YOLO
from tqdm import tqdm # เพิ่ม Progress bar ให้ดูง่ายขึ้น

# ==========================================
# 1. ตั้งค่ากฎ (Rules) ตาม Paper
# ==========================================

# 🔴 ปรับลด Threshold ลงเพื่อให้ YOLO กวาดไข่มาให้เยอะที่สุด แล้วเดี๋ยวให้ DINOv3 ฟันธง
DEFAULT_CONF = 0.15 
CLASS_CONF_THRESHOLDS = {
    0: 0.25, 
    1: 0.15, 
    2: 0.20,
}

LARGE_IMAGE_PIXELS = 1000 * 1000  
SMALL_IMAGE_PIXELS = 500 * 500    

MAX_AREA_RATIO_FOR_LARGE_IMG = 0.4  
MIN_AREA_RATIO_FOR_SMALL_IMG = 0.01 

# ==========================================
# 2. ฟังก์ชันหลัก
# ==========================================

def predict_and_filter(model_path, image_folder, output_folder, sample_csv_path):
    model = YOLO(model_path)
    os.makedirs(output_folder, exist_ok=True)

    # 🔴 อ่านชื่อไฟล์จากแม่แบบ จะได้ไม่ทำนายเกินหรือขาด
    df_sample = pd.read_csv(sample_csv_path)

    for img_name in tqdm(df_sample['filename'], desc="✂️ กำลังตัดรูปไข่พยาธิ"):
        img_path = os.path.join(image_folder, img_name)
        
        # ข้ามถ้ารูปไม่มีอยู่จริง
        if not os.path.exists(img_path):
            continue
            
        img_cv2 = cv2.imread(img_path)
        img_h, img_w = img_cv2.shape[:2]
        img_area = img_w * img_h

        results = model.predict(source=img_path, augment=True, conf=0.01, verbose=False)
        boxes = results[0].boxes
        
        valid_crops_count = 0

        for box in boxes:
            cls_id = int(box.cls[0].item())
            conf = box.conf[0].item()
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            
            box_area = (x2 - x1) * (y2 - y1)
            area_ratio = box_area / img_area

            # กรองความมั่นใจ
            required_conf = CLASS_CONF_THRESHOLDS.get(cls_id, DEFAULT_CONF)
            if conf < required_conf: continue

            # กรองขนาด
            if img_area > LARGE_IMAGE_PIXELS and area_ratio > MAX_AREA_RATIO_FOR_LARGE_IMG: continue
            if img_area < SMALL_IMAGE_PIXELS and area_ratio < MIN_AREA_RATIO_FOR_SMALL_IMG: continue

            # 🔴 ล็อกพิกัดกล่องไม่ให้ทะลุขอบภาพ (ป้องกัน OpenCV Error)
            x1, x2 = max(0, x1), min(img_w, x2)
            y1, y2 = max(0, y1), min(img_h, y2)

            crop_img = img_cv2[y1:y2, x1:x2]
            
            if crop_img.size == 0: continue

            # เซฟไฟล์
            crop_filename = f"{os.path.splitext(img_name)[0]}_cls{cls_id}_conf{conf:.2f}_{valid_crops_count}.jpg"
            save_path = os.path.join(output_folder, crop_filename)
            cv2.imwrite(save_path, crop_img)
            
            valid_crops_count += 1

if __name__ == "__main__":
    BEST_MODEL_PATH = r"D:\sup_ai\parasite-cv\runs\detect\train-11\weights\best.pt"  
    TEST_IMAGES_DIR = r"D:\sup_ai\parasite-cv\test_set\test"          
    CROPS_OUTPUT_DIR = r"D:\sup_ai\parasite-cv\cropped_eggs" 
    SAMPLE_CSV = r"D:\sup_ai\parasite-cv\sample_submission.csv" # 🔴 ใส่พาทไฟล์ submission แม่แบบ
    
    print("🚀 เริ่มการสกัดรูปไข่ด้วย YOLO")
    predict_and_filter(BEST_MODEL_PATH, TEST_IMAGES_DIR, CROPS_OUTPUT_DIR, SAMPLE_CSV)

### submit files

In [ ]:
import os
import cv2
import torch
import pandas as pd
from PIL import Image
from torchvision import transforms
from ultralytics import YOLO
from tqdm import tqdm

def create_final_submission(yolo_path, dino_path, test_dir, sample_csv, output_csv):
    print("Model Inference: YOLO + DINOv3")
    
    # 1. โหลด YOLO (Stage 1)
    yolo_model = YOLO(yolo_path)
    
    # 2. โหลด DINOv3 (Stage 2) พร้อมโหลดน้ำหนัก (Weights) ที่เทรนเสร็จแล้ว
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dino_model = DualStreamDINOv3(num_classes=11, freeze_backbone=False)
    dino_model.load_state_dict(torch.load(dino_path, map_location=device))
    dino_model.to(device)
    dino_model.eval() # เปิดโหมดใช้งานจริง (ไม่เทรนแล้ว)
    
    # ตัวแปลงภาพสำหรับ DINO
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    # 3. เตรียมไฟล์ Submission
    df_sample = pd.read_csv(sample_csv)
    results_list = []
    
    print("Infernce Working")
    for img_name in tqdm(df_sample['filename']):
        img_path = os.path.join(test_dir, img_name)
        best_label = -1 # ค่าเริ่มต้นถ้าไม่เจอไข่
        
        if os.path.exists(img_path):
            img_cv2 = cv2.imread(img_path)
            
            # --- Stage 1: YOLO หาระยะไข่ ---
            results = yolo_model.predict(source=img_cv2, augment=True, conf=0.01, verbose=False)
            boxes = results[0].boxes
            
            # ถ้า YOLO เจอกล่องไข่
            if len(boxes) > 0:
                # เลือกกล่องที่มั่นใจที่สุด
                best_box = max(boxes, key=lambda b: b.conf[0].item())
                x1, y1, x2, y2 = map(int, best_box.xyxy[0].tolist())
                
                # ป้องกันกล่องทะลุขอบจอ
                y1, y2 = max(0, y1), min(img_cv2.shape[0], y2)
                x1, x2 = max(0, x1), min(img_cv2.shape[1], x2)
                
                crop_cv2 = img_cv2[y1:y2, x1:x2]
                
                # แปลงจาก OpenCV (BGR) เป็น PIL (RGB) เพื่อส่งให้ DINO
                full_pil = Image.fromarray(cv2.cvtColor(img_cv2, cv2.COLOR_BGR2RGB))
                
                if crop_cv2.size > 0:
                    crop_pil = Image.fromarray(cv2.cvtColor(crop_cv2, cv2.COLOR_BGR2RGB))
                else:
                    crop_pil = full_pil # ถ้ากล่องผิดพลาด ให้ใช้รูปเต็มแทนเพื่อไม่ให้โค้ดพัง
                
                # --- Stage 2: DINOv3 ฟันธงคลาส ---
                full_tensor = transform(full_pil).unsqueeze(0).to(device)
                crop_tensor = transform(crop_pil).unsqueeze(0).to(device)
                
                with torch.no_grad(): # ปิดการจำ Gradient เพราะไม่ได้เทรนแล้ว
                    outputs = dino_model(full_tensor, crop_tensor)
                    best_label = torch.argmax(outputs, dim=1).item() # ดึงคลาสที่คะแนนสูงสุด
        
        results_list.append({
            "filename": img_name,
            "label": best_label
        })
        
    # 4. บันทึกผลลัพธ์
    df_out = pd.DataFrame(results_list)
    df_out.to_csv(output_csv, index=False)
    print(f"\n✅ ปิดจ็อบสำเร็จ! ไฟล์พร้อมส่งอยู่ที่: {output_csv}")

if __name__ == "__main__":
    # ตอนเทรน DINO เสร็จ อย่าลืมเซฟไฟล์ weights ด้วยคำสั่งนี้ในสคริปต์เทรนนะครับ:
    # torch.save(model.state_dict(), 'dino_best.pth')
    
    YOLO_WEIGHTS = r"runs\detect\train\weights\best.pt"
    DINO_WEIGHTS = "dino_best.pth" # ไฟล์ที่คุณเซฟมาจากการเทรน Stage 2
    TEST_DIR = r"D:\sup_ai\parasite-cv\test\images"
    SAMPLE_CSV = r"D:\sup_ai\parasite-cv\sample_submission.csv"
    OUTPUT_CSV = "submission_final_yolo_dino.csv"
    
    create_final_submission(YOLO_WEIGHTS, DINO_WEIGHTS, TEST_DIR, SAMPLE_CSV, OUTPUT_CSV)

# post and csv

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import pandas as pd
from ultralytics import YOLO
import timm
from tqdm import tqdm

# ==========================================
# 1. ตั้งค่ากฎการคัดกรองกล่อง YOLO (Rules)
# ==========================================
DEFAULT_CONF = 0.15 
CLASS_CONF_THRESHOLDS = {
    0: 0.25, 
    1: 0.15, 
    2: 0.20,
}

LARGE_IMAGE_PIXELS = 1000 * 1000  
SMALL_IMAGE_PIXELS = 500 * 500    

MAX_AREA_RATIO_FOR_LARGE_IMG = 0.4  
MIN_AREA_RATIO_FOR_SMALL_IMG = 0.001 

# ==========================================
# 2. ตั้งค่าเกณฑ์ความมั่นใจของ DINO
# ==========================================
# หาก DINO มั่นใจน้อยกว่าเปอร์เซ็นต์นี้ จะตอบ "-1" (ไม่ใช่พยาธิ)
# ขยับขึ้นจาก 0.35 มาเป็นค่ากลางๆ
DINO_CONF_THRESHOLD = 0.48

# ==========================================
# 3. Class DINOv2 (โครงสร้างต้องตรงกับตอนเทรน)
# ==========================================
class DINOClassifier(nn.Module):
    def __init__(self, num_classes=11):
        super().__init__()
        self.backbone = timm.create_model('vit_small_patch14_dinov2.lvd142m', pretrained=False)
        embed_dim = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

# ==========================================
# 4. ฟังก์ชันหลัก (Inference & Post-Processing)
# ==========================================
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 กำลังรันบนอุปกรณ์: {device}")

    # --- 🔴 กำหนด Path ทั้งหมด (แก้ไขให้ตรงกับเครื่องของคุณ) ---
    YOLO_WEIGHTS = r"D:\sup_ai\parasite-cv\runs\detect\train-11\weights\best.pt"
    DINO_WEIGHTS = r"D:\sup_ai\parasite-cv\dino_v2_timm_best.pth"
    CLASS_NAMES_FILE = r"D:\sup_ai\parasite-cv\class_names.txt"
    TEST_IMAGES_DIR = r"D:\sup_ai\parasite-cv\test_set\test"
    SAMPLE_CSV = r"D:\sup_ai\parasite-cv\sample_submission.csv"
    OUTPUT_CSV = r"D:\sup_ai\parasite-cv\submission_test_dinov2.csv"

    # --- โหลดรายชื่อคลาส ---
    with open(CLASS_NAMES_FILE, "r", encoding="utf-8") as f:
        class_names = [line.strip() for line in f.readlines()]
    num_classes = len(class_names)
    print(f"✅ โหลดรายชื่อคลาสสำเร็จ: จำนวน {num_classes} คลาส")

    # --- โหลดโมเดล YOLO และ DINO ---
    print("📥 กำลังโหลดโมเดล YOLOv8...")
    yolo_model = YOLO(YOLO_WEIGHTS)

    print("📥 กำลังโหลดโมเดล DINOv2...")
    dino_model = DINOClassifier(num_classes=num_classes)
    dino_model.load_state_dict(torch.load(DINO_WEIGHTS, map_location=device))
    dino_model.to(device)
    dino_model.eval()

    # --- Transform สำหรับป้อนเข้า DINO ---
    test_transforms = transforms.Compose([
        transforms.Resize((518, 518)), 
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    # --- โหลดไฟล์แม่แบบ Sample Submission ---
    print("\n📝 กำลังเตรียมไฟล์แม่แบบสำหรับการส่ง...")
    df_sample = pd.read_csv(SAMPLE_CSV)
    
    # 🔴 ป้องกัน Error "LossySetitemError" ด้วยการแปลงเป็น String
    df_sample['label'] = df_sample['label'].astype(str)

    print(f"🔥 เริ่มประมวลผลรูปทดสอบทั้งหมด {len(df_sample)} รูป...\n")
    
    # วนลูปประมวลผลตามรายชื่อไฟล์ใน sample_submission 
    for index, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="✂️ สแกนและทายผล"):
        img_name = row['filename']
        img_path = os.path.join(TEST_IMAGES_DIR, img_name)
        
        # ถ้ารูปไหนหายไปจากโฟลเดอร์ ให้ตอบ -1 ทันที
        if not os.path.exists(img_path):
            df_sample.at[index, 'label'] = "-1"
            continue
            
        img_pil = Image.open(img_path).convert("RGB")
        img_w, img_h = img_pil.size
        img_area = img_w * img_h

        # 1. ให้ YOLO หาตำแหน่งปรสิต
        results = yolo_model.predict(source=img_path, augment=True, conf=0.20, verbose=False)
        boxes = results[0].boxes
        
        valid_boxes = []

        # 2. นำกล่องมาคัดกรองตามกฎ
        for box in boxes:
            cls_id = int(box.cls[0].item())
            conf = box.conf[0].item()
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            
            box_area = (x2 - x1) * (y2 - y1)
            area_ratio = box_area / img_area

            # กรองความมั่นใจ YOLO
            required_conf = CLASS_CONF_THRESHOLDS.get(cls_id, DEFAULT_CONF)
            if conf < required_conf: continue

            # กรองขนาด (ป้องกันกล่องใหญ่เกินหรือเล็กเกิน)
            if img_area > LARGE_IMAGE_PIXELS and area_ratio > MAX_AREA_RATIO_FOR_LARGE_IMG: continue
            if img_area < SMALL_IMAGE_PIXELS and area_ratio < MIN_AREA_RATIO_FOR_SMALL_IMG: continue
            
            valid_boxes.append((box, conf))

        # 3. เตรียมรูปป้อนเข้า DINO
        if len(valid_boxes) > 0:
            # เลือกกล่องที่มั่นใจที่สุดที่ผ่านกฎ
            valid_boxes.sort(key=lambda x: x[1], reverse=True)
            best_box = valid_boxes[0][0]
            x1, y1, x2, y2 = map(int, best_box.xyxy[0].tolist())
            x1, x2 = max(0, x1), min(img_w, x2)
            y1, y2 = max(0, y1), min(img_h, y2)
            
            crop_img = img_pil.crop((x1, y1, x2, y2))
            input_tensor = test_transforms(crop_img).unsqueeze(0).to(device)
            
        elif len(boxes) > 0:
            # มีกล่องแต่ไม่ผ่านกฎเลย (ขยะแน่ๆ) -> เอากล่องที่มี conf สูงสุดมาเผื่อฟลุค
            best_box = sorted(boxes, key=lambda x: x.conf[0].item(), reverse=True)[0]
            x1, y1, x2, y2 = map(int, best_box.xyxy[0].tolist())
            x1, x2 = max(0, x1), min(img_w, x2)
            y1, y2 = max(0, y1), min(img_h, y2)
            crop_img = img_pil.crop((x1, y1, x2, y2))
            input_tensor = test_transforms(crop_img).unsqueeze(0).to(device)
            
        else:
            # YOLO ไม่เจออะไรเลย -> ส่งรูปเต็ม
            input_tensor = test_transforms(img_pil).unsqueeze(0).to(device)

        # 4. ให้ DINO ทายคลาสพร้อมกรองความมั่นใจ
        with torch.no_grad():
            outputs = dino_model(input_tensor)
            
            # แปลงคะแนนดิบเป็นความมั่นใจ (%)
            probabilities = F.softmax(outputs, dim=1)
            max_prob, best_index = torch.max(probabilities, dim=1)
            
            max_prob = max_prob.item()
            best_index = best_index.item()
            
            # 5. ตัดสินใจ (ถ้าไม่มั่นใจ ตอบ -1)
            if max_prob >= DINO_CONF_THRESHOLD:
                predicted_class = class_names[best_index]
            else:
                predicted_class = "-1"

        # 6. บันทึกคำตอบลง DataFrame (เขียนทับของเดิม)
        df_sample.at[index, 'label'] = str(predicted_class)

    # ==========================================
    # 5. บันทึกเป็น CSV สำหรับ Submit
    # ==========================================
    # 🔴 บังคับเลือกมาแค่ 2 คอลัมน์ที่ Kaggle ต้องการ
    final_submission = df_sample[['filename', 'label']].copy()
    
    # เติมค่าที่ว่างเป็น -1 กันเหนียว
    final_submission['label'] = final_submission['label'].fillna("-1")
    
    # เซฟไฟล์
    final_submission.to_csv(OUTPUT_CSV, index=False)
    
    # ==========================================
    # 6. สรุปผลลัพธ์ (Summary Report)
    # ==========================================
    print(f"\n✅ ปิดจ็อบสำเร็จ! บันทึกไฟล์ที่: {OUTPUT_CSV}")
    print("\n" + "="*50)
    print("📊 สรุปจำนวนการทำนายของแต่ละคลาส (Class Distribution):")
    print("="*50)
    
    counts = final_submission['label'].value_counts(dropna=False)
    print(counts.to_string())
    
    print("-" * 50)
    print(f"รวมรูปภาพทั้งหมดที่ประมวลผล: {len(final_submission)} รูป")
    print("="*50)

### yolo

In [ ]:
import os
import cv2
import pandas as pd
from ultralytics import YOLO
from tqdm import tqdm

# ==========================================
# 1. กฎการคัดกรองขยะ (YOLO Filtering Rules)
# ==========================================
# กรองด้วยความมั่นใจ (Confidence)
DEFAULT_CONF = 0.15 
CLASS_CONF_THRESHOLDS = {
    0: 0.25, 
    1: 0.15, 
    2: 0.20,
}

# กรองด้วยขนาด (Area Ratio) ป้องกันการตีกรอบฝุ่นหรือตีกรอบทั้งภาพ
LARGE_IMAGE_PIXELS = 1000 * 1000  
SMALL_IMAGE_PIXELS = 500 * 500    
MAX_AREA_RATIO_FOR_LARGE_IMG = 0.4  
MIN_AREA_RATIO_FOR_SMALL_IMG = 0.001 

if __name__ == "__main__":
    print("🚨 โหมดไม้ตายสุดท้าย: YOLOv8 ล้วน + TTA + Rule Filtering 🚨")
    
    # --- 🔴 ตรวจสอบ Path ให้ตรงกับเครื่องของคุณ ---
    YOLO_WEIGHTS = r"D:\sup_ai\parasite-cv\runs\detect\train-11\weights\best.pt"
    TEST_IMAGES_DIR = r"D:\sup_ai\parasite-cv\test_set\test"
    SAMPLE_CSV = r"D:\sup_ai\parasite-cv\sample_submission.csv"
    OUTPUT_CSV = r"D:\sup_ai\parasite-cv\submission_ULTIMATE_YOLO.csv"

    # โหลด YOLO Model
    print("📥 กำลังโหลดโมเดล YOLOv8...")
    yolo_model = YOLO(YOLO_WEIGHTS)

    # โหลดไฟล์แม่แบบและเตรียมคอลัมน์
    df_sample = pd.read_csv(SAMPLE_CSV)
    df_sample['label'] = df_sample['label'].astype(str)

    print(f"🔥 เริ่มสแกนและฟันธงข้อสอบ {len(df_sample)} รูป...\n")
    
    for index, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="🔍 สแกนด้วย YOLO"):
        img_name = row['filename']
        img_path = os.path.join(TEST_IMAGES_DIR, img_name)
        
        # ป้องกันไฟล์หาย
        if not os.path.exists(img_path):
            df_sample.at[index, 'label'] = "-1"
            continue

        # ใช้ OpenCV อ่านภาพเพื่อหาขนาดพิกเซลไปเข้าสูตร Area Ratio
        img_cv2 = cv2.imread(img_path)
        img_h, img_w = img_cv2.shape[:2]
        img_area = img_w * img_h

        # 🚀 ให้ YOLO สแกน (เปิด augment=True เพิ่มความแม่นยำขั้นสุด / conf=0.01 ดึงกล่องมาก่อน)
        results = yolo_model.predict(source=img_path, augment=True, conf=0.01, verbose=False)
        boxes = results[0].boxes
        
        valid_boxes = []

        # นำกล่องที่ได้มาเข้าเครื่องกรอง
        for box in boxes:
            cls_id = int(box.cls[0].item())
            conf = box.conf[0].item()
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            
            box_area = (x2 - x1) * (y2 - y1)
            area_ratio = box_area / img_area

            # ด่านที่ 1: กรองความมั่นใจ
            required_conf = CLASS_CONF_THRESHOLDS.get(cls_id, DEFAULT_CONF)
            if conf < required_conf: continue

            # ด่านที่ 2: กรองขนาด 
            if img_area > LARGE_IMAGE_PIXELS and area_ratio > MAX_AREA_RATIO_FOR_LARGE_IMG: continue
            if img_area < SMALL_IMAGE_PIXELS and area_ratio < MIN_AREA_RATIO_FOR_SMALL_IMG: continue
            
            # ถ้าผ่านด่านมาได้ ให้เก็บความมั่นใจและชื่อคลาสไว้
            valid_boxes.append((conf, cls_id))

        # 🎯 ตัดสินใจให้คำตอบ
        if len(valid_boxes) > 0:
            # กรณีเจอไข่พยาธิ: เรียงลำดับและเลือกกล่องที่มัน "มั่นใจที่สุด" 
            valid_boxes.sort(key=lambda x: x[0], reverse=True)
            best_cls = valid_boxes[0][1] # ดึงแค่ cls_id มาตอบเลย
            df_sample.at[index, 'label'] = str(best_cls)
        else:
            # กรณีหาไม่เจอเลย หรือ โดนกรองทิ้งหมด (ฟองอากาศ/ขยะ): ตอบ -1
            df_sample.at[index, 'label'] = "-1"

    # ==========================================
    # 2. บันทึกผลลัพธ์สุดท้าย
    # ==========================================
    final_submission = df_sample[['filename', 'label']].copy()
    final_submission['label'] = final_submission['label'].fillna("-1")
    final_submission.to_csv(OUTPUT_CSV, index=False)
    
    print(f"\n✅ ปิดจ็อบสำเร็จ! บันทึกไฟล์ที่: {OUTPUT_CSV}")
    print("\n" + "="*50)
    print("📊 สรุปจำนวนการทำนายของแต่ละคลาส (YOLO Only):")
    print("="*50)
    counts = final_submission['label'].value_counts(dropna=False)
    print(counts.to_string())
    print("="*50)

### yolo with no filter

In [ ]:
import os
import pandas as pd
from ultralytics import YOLO
from tqdm import tqdm

if __name__ == "__main__":
    print("🚨 โหมดไม้ตายสุดท้าย (ปลดล็อกพลัง 100%): YOLOv8 ไม่มี Filter 🚨")
    
    YOLO_WEIGHTS = r"D:\sup_ai\parasite-cv\runs\detect\train-11\weights\best.pt"
    TEST_IMAGES_DIR = r"D:\sup_ai\parasite-cv\test_set\test"
    SAMPLE_CSV = r"D:\sup_ai\parasite-cv\sample_submission.csv"
    OUTPUT_CSV = r"D:\sup_ai\parasite-cv\submission_YOLO_NO_FILTER.csv"

    yolo_model = YOLO(YOLO_WEIGHTS)
    df_sample = pd.read_csv(SAMPLE_CSV)
    df_sample['label'] = df_sample['label'].astype(str)

    # 🔴 ปรับระดับความกล้าของ YOLO (เปลี่ยนตัวเลขตรงนี้เพื่อหาจุด Sweet Spot)
    # ถ้าใส่ 0.05 แล้วยอด -1 น้อยไป (เช่นได้ 100 รูป) ให้ลองขยับเป็น 0.08 หรือ 0.10
    # เป้าหมายของเราคือทำให้ยอด -1 ออกมาอยู่แถวๆ 300 - 350 รูปครับ!
    YOLO_CONFIDENCE = 0.08 

    for index, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
        img_name = row['filename']
        img_path = os.path.join(TEST_IMAGES_DIR, img_name)
        
        if not os.path.exists(img_path):
            df_sample.at[index, 'label'] = "-1"
            continue

        # ให้ YOLO สแกน (เอาแค่ conf ขั้นต่ำ ไม่สน Area Ratio แล้ว)
        results = yolo_model.predict(source=img_path, augment=True, conf=YOLO_CONFIDENCE, verbose=False)
        boxes = results[0].boxes
        
        if len(boxes) > 0:
            # ไม่สนกฎอะไรทั้งนั้น เอาตัวที่ YOLO มั่นใจที่สุดมาตอบเลย!
            best_box = sorted(boxes, key=lambda x: x.conf[0].item(), reverse=True)[0]
            best_cls = int(best_box.cls[0].item())
            df_sample.at[index, 'label'] = str(best_cls)
        else:
            # ถ้า YOLO หาไม่เจอจริงๆ ค่อยตอบ -1
            df_sample.at[index, 'label'] = "-1"

    # ==========================================
    # เซฟไฟล์
    # ==========================================
    final_submission = df_sample[['filename', 'label']].copy()
    final_submission['label'] = final_submission['label'].fillna("-1")
    final_submission.to_csv(OUTPUT_CSV, index=False)
    
    print("\n" + "="*50)
    print("📊 สรุปจำนวนการทำนายของแต่ละคลาส (YOLO No Filter):")
    print("="*50)
    counts = final_submission['label'].value_counts(dropna=False)
    print(counts.to_string())
    print("="*50)

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import ipywidgets as widgets
from IPython.display import display

# ==========================================
# 1. ตั้งค่า Path
# ==========================================
YOLO_WEIGHTS = r"D:\sup_ai\parasite-cv\runs\detect\train-11\weights\best.pt"
TEST_IMAGES_DIR = r"D:\sup_ai\parasite-cv\test_set\test"

print("📥 กำลังโหลดโมเดล YOLOv8...")
model = YOLO(YOLO_WEIGHTS)

image_files = [f for f in os.listdir(TEST_IMAGES_DIR) if f.endswith(('.png', '.jpg', '.jpeg', '.JPG'))]

if len(image_files) == 0:
    print("❌ ไม่พบรูปภาพในโฟลเดอร์ที่ระบุ กรุณาตรวจสอบ Path อีกครั้ง")
else:
    print("✅ โหลดรูปภาพสำเร็จ! ลองเลื่อนสไลเดอร์ด้านล่างได้เลยครับ 👇")

# ==========================================
# 2. ฟังก์ชันหลักสำหรับอัปเดตภาพ
# ==========================================
def simulate_yolo(image_index, conf_percent, contrast, brightness):
    img_name = image_files[image_index]
    img_path = os.path.join(TEST_IMAGES_DIR, img_name)
    orig_img = cv2.imread(img_path)
    
    if orig_img is None:
        return
        
    # แปลงค่าจากสไลเดอร์
    alpha = contrast / 10.0
    beta = brightness
    conf_val = max(0.01, conf_percent / 100.0)
    
    # ปรับแสงสี
    img_processed = cv2.convertScaleAbs(orig_img, alpha=alpha, beta=beta)
    
    # ให้ YOLO สแกน (เปิด augment=True แบบที่ใช้ตอนส่งจริง)
    results = model.predict(img_processed, augment=True, conf=conf_val, verbose=False)
    annotated_img = results[0].plot()
    
    # แปลงระบบสีจาก BGR (OpenCV) เป็น RGB (Matplotlib)
    annotated_img = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
    
    # แสดงผลภาพ
    plt.figure(figsize=(14, 10))
    plt.imshow(annotated_img)
    plt.title(f"File: {img_name} | Conf: {conf_val:.2f} | Contrast: {alpha:.1f} | Brightness: {beta}", fontsize=14)
    plt.axis('off')
    plt.show()

# ==========================================
# 3. สร้าง UI สไลเดอร์ใน Jupyter Notebook
# ==========================================
widgets.interact(simulate_yolo, 
                 image_index=widgets.IntSlider(min=0, max=len(image_files)-1, step=1, value=0, description='📷 เปลี่ยนรูป:', layout=widgets.Layout(width='500px')),
                 conf_percent=widgets.IntSlider(min=1, max=100, step=1, value=8, description='🎯 Conf %:', layout=widgets.Layout(width='500px')),
                 contrast=widgets.IntSlider(min=1, max=30, step=1, value=10, description='🌗 Contrast:', layout=widgets.Layout(width='500px')),
                 brightness=widgets.IntSlider(min=-100, max=100, step=1, value=0, description='☀️ Bright:', layout=widgets.Layout(width='500px')))